# RR reconstruction and classical baselines

Kernel PCA, one-class SVM, the KPCA classifier, and the autoencoder now share loading, splits, artifact output, and evaluation. The autoencoder still trains on both classes by default, but its reported scores now use held-out records.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Inspect experiment settings


In [ ]:
import pandas as pd
from ecg_afib.config import load_config

PRESETS = ["rr_kernel_pca", "rr_one_class_svm", "rr_kpca_classifier", "rr_autoencoder"]
configs = [load_config(ROOT / "configs" / f"{name}.toml") for name in PRESETS]
pd.DataFrame([{"experiment": c.name, "method": c.method, "dataset": c.paths.dataset,
               "normal_only_autoencoder": c.model.normal_only, "output": c.paths.output} for c in configs])


## Run through the shared orchestration layer

The preset files contain model/training settings. No model classes or training loops are defined in this notebook. Existing runs are protected from overwrite.


In [ ]:
from ecg_afib.experiments import run_experiment

RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    for config in configs:
        run_experiment(config)
else:
    print("Set RUN_EXPERIMENTS=True after preparing the features, or use ecg-afib run --config ...")


## Inspect completed held-out results

All methods use mean recording scores and a validation-selected threshold. A larger reconstruction error is an AFib-oriented score, not proof that reconstruction can reliably distinguish the classes.


In [ ]:
import json
from ecg_afib.evaluation import compare_runs

completed = [c.path("output") for c in configs if (c.path("output") / "run.json").exists()
             and json.loads((c.path("output") / "run.json").read_text())["status"] == "complete"]
if completed:
    table = pd.DataFrame(compare_runs(completed))
else:
    table = pd.DataFrame()
    print("No completed baseline runs yet.")
table
